[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/hamilton-certified/notebooks/day-05-modules-namespacing.ipynb#scrollTo=dd000001)

---
# Day 5 · Modularity — Multiple Modules & Namespacing
**certified-journeys / hamilton-certified** · Day 5 · Modularity

> **Goal for today:** Split a pipeline across multiple Python modules, compose them with the Driver's `.with_modules()`, and use `subdag` to namespace reusable sub-pipelines.

In [ ]:
%pip install -q sf-hamilton

## Why Modularity Matters

As a pipeline grows, a single-file approach breaks down:

| Problem | Hamilton module solution |
|---|---|
| One file with 50+ functions | Split by responsibility: `raw_features.py`, `derived_features.py`, `model_inputs.py` |
| Two teams sharing a pipeline | Each team owns their module — no merge conflicts |
| Same transform on different columns | `subdag` — reuse a sub-pipeline with a namespace prefix |
| Conditional inclusion of a feature group | Omit a module from `.with_modules()` — that whole group disappears |

**The key insight:** Hamilton modules are just Python files. The discipline is one responsibility per file — treat them like microservices for your data.

In [ ]:
import sys, types
import numpy as np
import pandas as pd
from hamilton import driver
from hamilton.function_modifiers import tag, extract_columns, subdag
from hamilton.plugins import h_pandas

# Synthetic dataset — same as Day 4
rng = np.random.default_rng(42)
N = 300
raw_df = pd.DataFrame({
    'age':                   rng.integers(18, 75, N).astype(float),
    'tenure_months':         rng.integers(0, 72, N).astype(float),
    'monthly_spend':         rng.exponential(80, N),
    'num_products':          rng.integers(1, 6, N).astype(float),
    'has_complaint':         rng.integers(0, 2, N).astype(float),
    'days_since_last_login': rng.integers(0, 90, N).astype(float),
})
print('Dataset ready:', raw_df.shape)

## Step 1 · Module 1: `raw_features` — Data Ingestion Layer

The first module's job is purely **ingestion and extraction** — get the data in, expose each column as a named node. No transformations here.

In [ ]:
# ============================================================
# MODULE 1: raw_features.py
# Responsibility: ingest source data, expose columns as nodes
# ============================================================

@extract_columns(
    'age', 'tenure_months', 'monthly_spend',
    'num_products', 'has_complaint', 'days_since_last_login'
)
def raw_features(raw_data: pd.DataFrame) -> pd.DataFrame:
    """Source layer: extract all raw columns."""
    return raw_data[[
        'age', 'tenure_months', 'monthly_spend',
        'num_products', 'has_complaint', 'days_since_last_login'
    ]]

raw_module = types.ModuleType('raw_features')
raw_module.raw_features = raw_features
sys.modules['raw_features'] = raw_module
print('Module 1 (raw_features) defined.')

## Step 2 · Module 2: `derived_features` — Transformation Layer

The second module takes raw Series as inputs and produces transformed features. It **depends on** Module 1's nodes by name — no import required. Hamilton wires them automatically at Driver build time.

In [ ]:
# ============================================================
# MODULE 2: derived_features.py
# Responsibility: transform raw Series into ML-ready features
# Depends on: raw_features module (by node name)
# ============================================================

@tag(feature_type='numerical')
def age_zscore(age: pd.Series) -> pd.Series:
    return (age - age.mean()) / age.std()

@tag(feature_type='numerical')
def tenure_years(tenure_months: pd.Series) -> pd.Series:
    return tenure_months / 12.0

@tag(feature_type='numerical')
def monthly_spend_log(monthly_spend: pd.Series) -> pd.Series:
    return np.log1p(monthly_spend)

@tag(feature_type='numerical')
def recency_score(days_since_last_login: pd.Series) -> pd.Series:
    return 1.0 - (days_since_last_login / days_since_last_login.max())

@tag(feature_type='boolean')
def is_high_spender(monthly_spend: pd.Series) -> pd.Series:
    return (monthly_spend > monthly_spend.quantile(0.75)).astype(float)

@tag(feature_type='boolean')
def is_new_customer(tenure_months: pd.Series) -> pd.Series:
    return (tenure_months < 6).astype(float)

derived_module = types.ModuleType('derived_features')
for fn in [age_zscore, tenure_years, monthly_spend_log, recency_score,
           is_high_spender, is_new_customer]:
    setattr(derived_module, fn.__name__, fn)
sys.modules['derived_features'] = derived_module
print('Module 2 (derived_features) defined.')

## Step 3 · Module 3: `model_inputs` — Assembly Layer

The third module assembles derived features into model-ready signals. It depends on nodes from **both** Module 1 and Module 2 — and it doesn't need to import either.

In [ ]:
# ============================================================
# MODULE 3: model_inputs.py
# Responsibility: interaction features + final model vector
# Depends on: derived_features module (by node name)
# ============================================================

@tag(feature_type='numerical')
def spend_x_tenure(monthly_spend_log: pd.Series, tenure_years: pd.Series) -> pd.Series:
    """Interaction: log-spend × tenure."""
    return monthly_spend_log * tenure_years

@tag(feature_type='numerical')
def engagement_score(recency_score: pd.Series, num_products: pd.Series) -> pd.Series:
    """Engagement: recency × normalised product breadth."""
    return recency_score * (num_products / num_products.max())

model_module = types.ModuleType('model_inputs')
for fn in [spend_x_tenure, engagement_score]:
    setattr(model_module, fn.__name__, fn)
sys.modules['model_inputs'] = model_module
print('Module 3 (model_inputs) defined.')

## Step 4 · Compose All Three Modules

Now we wire all three modules into a single Driver. The order in `.with_modules()` doesn't matter — Hamilton resolves dependencies by name across all modules.

In [ ]:
# Compose all three modules
dr = (
    driver.Builder()
    .with_modules(raw_module, derived_module, model_module)  # order doesn't matter
    .with_adapters(h_pandas.PandasDataFrameResult())
    .build()
)

print('Nodes from raw_module:    ', [v.name for v in dr.list_available_variables() if v.name in ['age','tenure_months','monthly_spend','num_products','has_complaint','days_since_last_login']])
print('Nodes from derived_module:', [v.name for v in dr.list_available_variables() if 'feature_type' in v.tags][:6])
print('Nodes from model_module:  ', ['spend_x_tenure', 'engagement_score'])

# Execute the full cross-module pipeline
MODEL_FEATURES = [
    'age_zscore', 'tenure_years', 'monthly_spend_log', 'recency_score',
    'is_high_spender', 'is_new_customer', 'spend_x_tenure', 'engagement_score',
]
feature_df = dr.execute(MODEL_FEATURES, inputs={'raw_data': raw_df})
print('\nFeature matrix shape:', feature_df.shape)
print(feature_df.head(3).round(3).to_string())

### What just happened?
- **Three modules, zero explicit imports** — `model_inputs` uses `monthly_spend_log` and `tenure_years` by name; Hamilton finds them in `derived_module` automatically.
- **Adding or removing a module** is a single `.with_modules()` change — the graph adapts.
- **The full DAG spans all three files** — Hamilton sees it as one graph.

## Step 5 · `subdag` — Reusable Namespaced Sub-Pipelines

`subdag` lets you **reuse a module under a different namespace prefix**. This is essential when you want the same transformation logic applied to different contexts — e.g. normalize `age` and `tenure` using the same z-score recipe, but as separate named nodes.

```python
@subdag(
    normalization_module,
    inputs={"value": source("age")},
    config={},
)
def age_features(value_zscore: pd.Series) -> pd.Series:
    ...
```

All nodes inside `normalization_module` are prefixed with the function name — so `zscore` becomes `age_features.zscore`, avoiding name collisions.

In [ ]:
from hamilton.function_modifiers import subdag, source, value

# A reusable normalization module — works on any Series named 'value'
def value_mean(value: pd.Series) -> float:
    return float(value.mean())

def value_std(value: pd.Series) -> float:
    s = float(value.std())
    return s if s > 0 else 1.0  # avoid divide-by-zero

def value_zscore(value: pd.Series, value_mean: float, value_std: float) -> pd.Series:
    """Z-score normalize 'value'."""
    return (value - value_mean) / value_std

norm_module = types.ModuleType('normalization')
for fn in [value_mean, value_std, value_zscore]:
    setattr(norm_module, fn.__name__, fn)
sys.modules['normalization'] = norm_module

# Apply the same normalization to two different columns via subdag
@tag(feature_type='numerical')
@subdag(
    norm_module,
    inputs={'value': source('age')},
    config={},
)
def age_normalized(value_zscore: pd.Series) -> pd.Series:
    """Age z-scored via the reusable normalization subdag."""
    return value_zscore

@tag(feature_type='numerical')
@subdag(
    norm_module,
    inputs={'value': source('monthly_spend')},
    config={},
)
def spend_normalized(value_zscore: pd.Series) -> pd.Series:
    """Monthly spend z-scored via the reusable normalization subdag."""
    return value_zscore

subdag_module = types.ModuleType('subdag_features')
subdag_module.age_normalized = age_normalized
subdag_module.spend_normalized = spend_normalized
sys.modules['subdag_features'] = subdag_module

dr_sub = (
    driver.Builder()
    .with_modules(raw_module, subdag_module)
    .build()
)

result = dr_sub.execute(['age_normalized', 'spend_normalized'], inputs={'raw_data': raw_df})
print('age_normalized (mean, std):', round(result['age_normalized'].mean(), 4), round(result['age_normalized'].std(), 4))
print('spend_normalized (mean, std):', round(result['spend_normalized'].mean(), 4), round(result['spend_normalized'].std(), 4))

### What just happened?
- **`subdag`** runs `norm_module` twice — once with `age` as the `value` input, once with `monthly_spend`.
- **The namespacing** (`age_normalized.*`, `spend_normalized.*`) prevents node name collisions.
- **One normalization module** used for N features — change the z-score logic once, it propagates everywhere.

## Step 6 · Dropping a Module — Conditional Feature Groups

A powerful consequence of the module model: if you omit a module from `.with_modules()`, all its nodes disappear from the graph. This is how you do **conditional feature inclusion** without touching any function code.

In [ ]:
# Build WITHOUT model_module — interaction features are excluded
dr_no_interactions = (
    driver.Builder()
    .with_modules(raw_module, derived_module)  # model_module intentionally omitted
    .with_adapters(h_pandas.PandasDataFrameResult())
    .build()
)

base_features = ['age_zscore', 'tenure_years', 'monthly_spend_log', 'recency_score',
                 'is_high_spender', 'is_new_customer']

base_df = dr_no_interactions.execute(base_features, inputs={'raw_data': raw_df})
print('Base pipeline (no interactions):', base_df.shape)

# Compare: full pipeline has interaction features
print('Full pipeline:', feature_df.shape)
print('\nDifference: interaction features spend_x_tenure and engagement_score are absent from base pipeline')

### What just happened?
- **Removing `model_module`** from `.with_modules()` is the only change — no code in the functions was touched.
- This pattern enables **A/B testing feature sets**: run two Drivers with different module combinations, compare model metrics.
- It also makes **staged rollouts** safe: add modules incrementally as features are validated.

In [ ]:
# Challenge: create a new module called `risk_features` with two functions:
# 1. `complaint_rate(has_complaint, tenure_years)` — complaints per year of tenure
# 2. `churn_risk_score(is_high_spender, recency_score, complaint_rate)` — composite risk
# Add it to the full Driver and verify both nodes appear in list_available_variables()

# def complaint_rate(has_complaint: pd.Series, tenure_years: pd.Series) -> pd.Series:
#     ...

# def churn_risk_score(...) -> pd.Series:
#     ...

print('Build the risk_features module and wire it into the Driver!')

---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| Module = one Python file | One responsibility: ingestion, derived features, model inputs |
| `.with_modules(m1, m2, m3)` | Order doesn't matter; Hamilton resolves cross-module deps by name |
| Omitting a module | Entire feature group removed from the graph — zero code changes |
| `subdag` | Reuse a sub-pipeline with a namespace prefix — avoids node collisions |
| `source('node_name')` | Maps a subdag input parameter to a named node in the outer graph |

> **Tip:** If your DAG visualization is too wide to read, your modules are too coupled. Use `subdag()` to introduce hierarchy.

---
## What's next
**Day 6** → ETL transformation patterns: `@does` for DRY logic, `@check_output` for data quality constraints, and running the same pipeline against multiple data sources.

Mark Day 5 complete in your [tracker](../index.html).